# 🦉 Strixlibe × Google Colab GPU LLM Server (Strata + Qwen3.8-Flash-Next 125B 超高知能版)
自律型ペンテストエージェント「Strixlibe」向けに、Google Colab（**NVIDIA L4 24GB GPU + ハイメモリ推奨**）上で、最新の階層オフロード推論エンジン **Strata** を使用して 1250億パラメータの超巨大 MoE モデル **Qwen3.8-Flash-Next** をホストするノートブックです。

### ⚡ 特徴と設計
- **1クリック完全自動完走**: 複数のセルを実行する必要なく、下のセルを 1 回実行するだけで「完全クリーン初期化 → 依存導入 → モデルDL → サーバー起動 → トンネル確立 → 疎通テスト」までノンストップで自動完走します。
- **常にまっさらな状態を保証**: 実行ごとに過去の残存プロセスや一時ログを完全強制キルし、いつでも同一のクリーンな環境で検証できます。
- **125B MoE を L4 (24GB) で駆動**: VRAM・ホストRAM・CPUに階層分散配置することで、本来 A100 複数枚が必要な 125B モデルを単一 GPU で駆動。

> **⚠️ ランタイム設定の注意**: Colab のメニュー「ランタイム」→「ランタイムのタイプを変更」で、**GPU（L4）** かつ **ハイメモリ（High-RAM: 約53GB）** を選択してください。

In [ ]:
#@title 🚀 1クリック完全自動起動（まっさらな状態から最後まで自動完走） { run: "auto" }
import os
import sys
import time
import psutil
import subprocess
import requests
import re

# ==================== 1. 設定項目 ====================
MODEL_FAMILY = "qwen" #@param ["qwen", "coder"]
QUANT_LEVEL = "IQ2_XS" #@param ["IQ2_XS", "Q2_0", "IQ3_XXS", "IQ3_S"]
CONTEXT_LENGTH = 32768 #@param [16384, 32768, 65536, 131072]
HF_TOKEN = "" #@param {type:"string"}
DISPATCHER_URL = "https://strix-dispatcher-2acg5ejyka-uw.a.run.app" #@param {type:"string"}
DISPATCHER_API_KEY = "" #@param {type:"string"}
SLACK_WEBHOOK_URL = "" #@param {type:"string"}
IDLE_TIMEOUT_MINUTES = 25 #@param {type:"integer"}
TARGET_ENV = "all" #@param ["all", "staging", "production"]

print("="*65)
print("🦉 Strixlibe × Strata 125B 自動起動パイプライン開始")
print("="*65)

# ==================== 2. スペック確認 & 完全初期化 ====================
print("\n[*] [1/5] ハードウェア & メモリ確認中...")
sys.stdout.flush()
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"[+] Total System RAM: {total_ram_gb:.1f} GB")
if total_ram_gb < 30:
    print("⚠️ [WARNING] システムRAMが30GB未満です。「ランタイムのタイプを変更」で「ハイメモリ」を推奨します。")

print("[*] 前回の残存プロセスと古いログを完全クリーンアップ中...")
subprocess.run("pkill -9 -f strata || true", shell=True)
subprocess.run("pkill -9 -f cloudflared || true", shell=True)
subprocess.run("rm -f /content/strata.log", shell=True)

# ==================== 3. 依存ライブラリ & Strata クローン ====================
print("\n[*] [2/5] 必要なツール・Strataリポジトリを準備中...")
sys.stdout.flush()
subprocess.run("apt-get install -y -qq git zstd curl pciutils python3-venv", shell=True)
subprocess.run("pip install -q --no-warn-conflicts hf-transfer", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# cloudflared 導入
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[*] cloudflared をダウンロード中...")
    subprocess.run("curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)

# Strata クローン
strata_dir = "/content/Strata"
if not os.path.exists(strata_dir):
    print("[*] Strata リポジトリをクローン中...")
    subprocess.run("git clone --depth 1 https://github.com/Niko1221/Strata.git /content/Strata", shell=True)
else:
    print("[*] 既存のStrataをクリーンリセット中...")
    subprocess.run("cd /content/Strata && git fetch origin && git reset --hard origin/main && git clean -fd", shell=True)

# ==================== 4. モデルダウンロード & CUDAキャッシュ構築 ====================
print(f"\n[*] [3/5] モデルダウンロード開始: Qwen3.8-Flash-Next ({QUANT_LEVEL}, ctx: {CONTEXT_LENGTH}) ...")
print("[*] 約40GBのモデル重みを高速ダウンロード中（所要時間: 約3〜6分）...")
print("[*] 画面に進捗が順次表示されます。そのままお待ちください...\n")
sys.stdout.flush()

setup_cmd = f"python3 setup.py --yes --family {MODEL_FAMILY} --model {QUANT_LEVEL} --context {CONTEXT_LENGTH} --host 0.0.0.0 --port 8080 --no-start"
ret = subprocess.run(setup_cmd, shell=True, cwd=strata_dir)
if ret.returncode != 0:
    print(f"[-] モデルセットアップで警告またはエラーが発生しました (code {ret.returncode})")

# ==================== 5. Strata サーバー & Cloudflare Tunnel 起動 ====================
print("\n[*] [4/5] Strata サーバーをバックグラウンド起動中...")
sys.stdout.flush()

launch_cmd = ["python3", f"{strata_dir}/setup.py", "--family", MODEL_FAMILY, "--model", QUANT_LEVEL, "--context", str(CONTEXT_LENGTH), "--host", "0.0.0.0", "--port", "8080", "--yes"]
strata_log = open("/content/strata.log", "w")
server_proc = subprocess.Popen(launch_cmd, cwd=strata_dir, stdout=strata_log, stderr=subprocess.STDOUT)

print("[*] 重みをGPUおよびホストRAMにロード中（約1〜2分）...")
ready = False
for i in range(90):
    try:
        r = requests.get("http://127.0.0.1:8080/health", timeout=1)
        if r.status_code == 200:
            ready = True
            break
    except:
        try:
            r = requests.get("http://127.0.0.1:8080/v1/models", timeout=1)
            if r.status_code == 200:
                ready = True
                break
        except:
            pass
    time.sleep(2)

if not ready:
    print("[-] Health check pending. /content/strata.log の末尾:")
    subprocess.run("tail -n 25 /content/strata.log || true", shell=True)
else:
    print("[+] Strata サーバーが正常起動しました（Port 8080）！")

print("[*] Cloudflare Tunnel を確立中...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--protocol", "http2", "--url", "http://localhost:8080", "--http-host-header", "localhost:8080"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if not tunnel_url:
    print("[-] トンネルURLの取得に失敗しました。ローカル接続のみ利用可能です。")
    api_base = "http://127.0.0.1:8080/v1"
else:
    api_base = f"{tunnel_url}/v1"
    print(f"\n🎉 ==========================================================")
    print(f"[+] Cloudflare Tunnel 確立成功！")
    print(f"[+] 公開 URL: {tunnel_url}")
    print(f"[+] OpenAI API Base: {api_base}")
    print(f"==========================================================\n")

ACTIVE_MODEL_NAME = f"qwen3.8-flash-next-{QUANT_LEVEL.lower()}"

# ==================== 6. 疎通テスト（125B ベンチマーク） ====================
print("[*] [5/5] 125B モデルへの疎通推論テストを実行中...")
test_payload = {
    "model": ACTIVE_MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Strixペンテストエージェントの125B推論基盤テストです。自律ペンテストにおいてあなたの得意とする役割を日本語で簡潔に2文以内で述べてください。"}
    ],
    "max_tokens": 150,
    "temperature": 0.6
}

try:
    resp = requests.post("http://127.0.0.1:8080/v1/chat/completions", json=test_payload, timeout=90)
    if resp.status_code == 200:
        result = resp.json()
        reply = result["choices"][0]["message"]["content"]
        print("\n[+] 🎉 125B 推論テスト大成功！モデルからの返答:")
        print("-" * 55)
        print(reply.strip())
        print("-" * 55)
    else:
        print(f"[-] テスト失敗: HTTP {resp.status_code} - {resp.text}")
except Exception as e:
    print(f"[-] テスト例外: {e}")

# Dispatcher 同期
if DISPATCHER_URL and DISPATCHER_API_KEY:
    try:
        reg_payload = {
            "provider": "colab-strata",
            "base_url": api_base,
            "model_name": ACTIVE_MODEL_NAME,
            "env_target": TARGET_ENV,
            "ttl_minutes": 120
        }
        headers = {"X-API-Key": DISPATCHER_API_KEY, "Content-Type": "application/json"}
        requests.post(f"{DISPATCHER_URL}/api/v1/llm-session", json=reg_payload, headers=headers, timeout=10)
        print("[+] Strixlibe Dispatcher に登録完了！")
    except Exception as e:
        print(f"[-] Dispatcher 登録スキップ: {e}")

print("\n[OK] 全工程が完了しました！Strixlibe からのリクエスト待機中。")


### 2. CU保護ウォッチドッグ（無通信時の自動停止）
※ 必要に応じてバックグラウンド監視を動かす場合は実行してください（放置時のCompute Unit消費を防止します）。

In [ ]:
import threading
import time
from google.colab import runtime

def watchdog_worker():
    print(f"[*] Watchdog background worker started. Auto-terminate after {IDLE_TIMEOUT_MINUTES}m inactivity.")
    last_seen_time = time.time()
    idle_timeout_seconds = IDLE_TIMEOUT_MINUTES * 60
    
    while True:
        time.sleep(30)
        elapsed = time.time() - last_seen_time
        if elapsed > idle_timeout_seconds:
            print(f"[!] Inactivity threshold exceeded. Terminating runtime to preserve Compute Units.")
            runtime.unassign()
            break

t = threading.Thread(target=watchdog_worker, daemon=True)
t.start()
print("[OK] Watchdog is active.")
